<!-- notebook-header -->
<a id="top"></a>
<a id="topic-stateful-api-design"></a>

# 45. Stateful API Design

*LeetCode Playbook · topic notebook 45* · Stateful API Design.

← [Quickselect](44_Quickselect.ipynb) · [Topic index](00_Topic_Index.ipynb#notebooks) · [A–Z problem finder](67_Problem_Finder.ipynb#finder) · [Logger Rate Limiter](46_Logger_Rate_Limiter.ipynb) →

Choose a linked lesson to open its runnable examples.

## Stateful API Design

> A design problem is a contract: a list of operations, each with a price ("O(1)", "O(log n)"). Give every operation a structure that answers its question at that price, write down the one sentence that keeps the structures in sync, and the methods nearly write themselves.

By the end of this section you will take a prompt like *"design a hit counter"*, a class that counts the hits of the last five minutes, and turn it into a working class inside 25 minutes. You will ask the questions that settle the requirements, give every operation a structure that answers it at the asked price, write the invariant that keeps those structures agreeing, and type the methods as calls to two or three small helpers.

Every class here is built by that one procedure, and the section ends with the script for saying it out loud.

**Reach for it when** the prompt says *Design ...* or *Implement a class ...*, describes a **tracker, counter, rate limiter, cache or store**, lists method signatures, or puts a cost on every call ("each in O(1)", "many queries after one construction").

**In this repo:** `design/` (22 problems) · bank: `practice/simple/24_lru_cache.py` · basics: `practice/simple/basics/stacks/01_array_stack_and_queue_via_two_stacks.py`.

Several design classes live with the structure they are built on. Min Stack, a stack that also reports its minimum in O(1) (`stack/min_stack.py`, `practice/simple/14_min_stack.py`), and `queues/number_of_recent_calls.py`, `queues/design_circular_queue.py`, `queues/design_circular_deque.py` and `queues/implement_queue_using_stacks.py` are in [Stacks & Queues](00_Topic_Index.ipynb#s07). MedianFinder, the running median of a stream (`heap/find_median_from_data_stream.py`, `practice/simple/34_find_median_from_data_stream.py`), `heap/design_twitter.py` and `heap/kth_largest_element_in_a_stream.py` are in [Heaps](15_Heaps.ipynb#topic-heaps).

The others sit in three more sections: `intervals/my_calendar_iii.py` in [Intervals & Sweep Line](16_Intervals_and_Sweep_Line.ipynb#topic-intervals-and-sweep-line); `tries/implement_trie_prefix_tree.py`, `tries/design_search_autocomplete_system.py` and `tries/stream_of_characters.py` in [Tries](14_Tries.ipynb#topic-tries); and `trees/serialize_and_deserialize_binary_tree.py` in [Trees](13_Trees.ipynb#topic-trees).

### The picture

LRU Cache is the showcase, so the picture is drawn for it. `get(key)` returns the value or −1 and `put(key, value)` stores one, both in O(1), and when the cache is full a `put` of a new key evicts the key that was used longest ago. The picture is the whole method on one problem: each operation becomes a question, each question gets a structure, and one sentence ties the structures together.

```text
THE METHOD, drawn for LRU Cache

  operation    price   the question it asks       the structure that answers it
  get(key)     O(1)    where is key's entry?      dict key -> node
                       make it the newest         list: unlink, push after HEAD
  put(key, v)  O(1)    where is key's entry?      dict;  make it the newest: list
  evict        O(1)    who is the oldest?         list: TAIL.prev;  forget it everywhere: del map[node.key]

  map   { 1: *, 2: *, 3: * }      each * points straight at a node in the list
  list  HEAD <-> [3] <-> [1] <-> [2] <-> TAIL
                 newest           oldest: evicted next
  the sentence that ties them: the keys in the dict are exactly the nodes in the list,
  and the list order is the recency order
```

Why it is fast: the brute force keeps one list of `(key, value)` pairs in recency order. Every `get` scans it to find the key, which is O(n), and moving the pair to the front shifts everything behind it, another O(n).

The two structures remove the two costs. The dict removes the scan, because it hands us the node directly. The doubly linked list removes the shift, because a node knows both of its neighbours, so unlinking it is two pointer writes and no other node moves.

### From idea to code

**The idea in one sentence:** *for each operation, ask which question it must answer and pick the structure that answers it at the required price; then write the invariant that keeps all the structures describing the same items.*

The seven decisions of [From Idea to Code](02_Idea_to_Code.ipynb#topic-idea-to-code) apply to a class as they apply to a loop: the items are now calls, the state lives in `self`, and every decision has a fixed place. **State** is the structures in `self`, one `# STATE` line in `__init__` for each, with its **Definition** as the comment, and **Init** is what those lines assign. The **Invariant** is the sentence that is true between any two calls, written as the last comment of `__init__`.

A **Step** is what a public method does to the state, and a **Fix** is the private helper that makes the invariant true again, such as `_expire` or `_evict`. **Record** is the running value a write keeps current, a total or a minimum, and **Return** is what each query hands back, including its answer when nothing matches.

The procedure below makes those decisions in the order you work, against a 25-minute clock. Each step leaves something written down, so you hold one decision in your head at a time, and the same procedure builds every class in this section.

1. **Requirements, minutes 0-3.** Ask before you design, and write the answers as the class docstring. Ask whether timestamps only increase, whether two events can share one, whether a write can correct an earlier one, what a query returns when nothing matches, which call is the most frequent, and whether the capacity is bounded and can be 0. Each answer changes a structure, so each is worth its sentence.
2. **Operations and costs, minutes 3-5.** List every public method with the price the prompt puts on it, or the price you propose when it names none. Then turn each into the question it must answer: `get(key)` asks "where is key's entry?", and eviction asks "who is the oldest?".
3. **One structure per question, minutes 5-7.** Pick each structure from the table below by its question and its price, never by the problem's name. One structure, usually a dict, holds the truth, and every other one is an index that answers one question fast. Write them as `# STATE` lines in `__init__`, each with its definition.
4. **The invariant, minutes 7-8.** Write the one sentence that is true between any two calls as the last comment of `__init__`. In a design it almost always reads "the indexes agree with the truth", and every method must leave it true.
5. **Helpers, minutes 8-10.** Name the repairs, `_expire(t)`, `_unlink(node)`, `_evict()`, and give each a one-line contract that says what is true after it runs. A helper is the invariant restored, with a name.
6. **Methods, minutes 10-18.** Write each public method as two or three helper calls, then fill in the helper bodies. After each method, say which helper restored the invariant.
7. **Trace and edge cases, minutes 18-25.** Trace four or five calls on paper, writing down the structures after each. Then say the edge cases out loud: the empty structure, capacity 0 and 1, a missing key, a key written twice, a burst at one timestamp. Close with the cost of each operation and the memory.

| Minutes | You are doing | You have written |
|---|---|---|
| 0-3 | asking the requirement questions | the class docstring |
| 3-7 | operation → question → structure, with costs | `__init__`, one `# STATE` line per structure |
| 7-10 | the invariant and the helper names | the last comment of `__init__`, the helper signatures |
| 10-18 | public methods as helper calls, then the helper bodies | the class |
| 18-25 | a trace of 4-5 calls, the edge cases, the costs | a state table and the closing sentence |

Every design makes one more choice: who pays, the write or the read? Keep the answer current on every write when reads must be O(1): a running total for the mean of the last k values, a count of ones for a bitset that flips, the smallest use count for a cache that evicts its rarest key, the latest timestamp for a price feed.

Leave the cleanup to the reads when writes must stay cheap: a hit counter expires old seconds when someone asks, and a heap drops stale tops when they surface. Say which one you chose; it is the sentence the interviewer is listening for.

Two words carry the method. The **source of truth** is the one structure that is always right, usually a dict: when structures disagree, the truth wins. An **index** is any extra structure kept only so that one question is fast, a heap for "what is the smallest?", a linked list for "who is the oldest?".

An index is updated eagerly, as LRU relinks a node on every use, or checked lazily, as a heap entry that disagrees with the dict is skipped. Either way the invariant is the same sentence, *the indexes agree with the truth*, and every class in this section is this one recipe.

**Lazy deletion** is the second way made precise: nothing leaves the heap at the moment it dies. The dict is corrected, the old heap entry stays behind as a **stale** entry, and whoever reads the heap's top first checks it against the dict and pops it if it disagrees.

Some prices in the table are **amortised**: one call may do a lot of work, such as popping many expired events, but every item is added once and removed at most once, so n calls cost O(n) in total, O(1) per call averaged over the run. The table is the lookup for step 3: find the question, then read off the structure and its price.

| I need to do this fast | The question it answers | Use | Cost |
|---|---|---|---|
| lookup by key, membership | "where is x? have I seen x?" | `dict` / `set` | O(1) |
| count occurrences | "how many x so far?" | `Counter` / `dict` | O(1) per update |
| min or max, with inserts | "what is the smallest now?" | heap ([Heaps](15_Heaps.ipynb#topic-heaps)) | O(log n) |
| min or max, with inserts and arbitrary deletes or corrections | "what is the smallest *live* item?" | heap + lazy deletion (skip stale tops) | O(log n) amortised |
| min or max of a sliding window | "what is the max of the last k?" | monotonic deque ([Sliding Window](07_Sliding_Window.ipynb#topic-sliding-window)) | O(1) amortised |
| next greater / previous smaller | "who resolves me?" | monotonic stack ([Monotonic Stack](10_Monotonic_Stack.ipynb#topic-monotonic-stack)) | O(1) amortised |
| median of a stream | "what is the middle?" | two heaps | O(log n) |
| events inside a time window | "how many in the last w seconds?" | `deque` + a running total | O(1) amortised |
| first position ≥ x in a sorted list | "where would x go?" | `bisect` ([Binary Search](11_Binary_Search.ipynb#topic-binary-search)) | O(log n) |
| "the value as of time t" | "what was the last write at or before t?" | per-key list of (t, value) + `bisect` | O(log n) |
| a sorted set of disjoint intervals | "which intervals touch x?" | sorted list(s) + `bisect` | O(log n) search, O(n) insert |
| range sum, no updates / with point updates | "what is sum(a[i:j])?" | prefix sums / Fenwick tree ([Prefix Sums](05_Prefix_Sums.ipynb#topic-prefix-sums)) | O(1) / O(log n) |
| "are a and b connected?" with merges | "same group?" | union-find | almost O(1) |
| prefix lookups among many words | "which words start with p?" | trie of dicts ([Tries](14_Tries.ipynb#topic-tries)) | O(length) |
| add and remove at both ends | "oldest? newest?" | `deque` | O(1) |
| undo, nesting, most recent | "what is the latest unresolved item?" | `list` as a stack | O(1) |
| lookup + order by recency | "which key was used longest ago?" | `dict` + doubly linked list, or `OrderedDict` | O(1) |
| a uniformly random element, with deletes | "pick any stored item" | dense `list` + `dict` value → index (swap with last) | O(1) |
| min or max of counts that change by ±1 | "which key has the smallest count?" | count buckets + a pointer to the min or max count | O(1) |

## Think in method calls, state, and invariants

An **API** here means the methods the caller can use, such as `hit(t)` and `getHits(t)`.
An **invariant** is a fact your stored state must satisfy after each operation.

```py
counter = HitCounter()    # One object; its state starts empty.
counter.hit(1)            # Changes that object's state.
counter.hit(2)            # Remembers the previous hit as well.
counter.getHits(3)        # Reads the accumulated state: 2.
```

`self.window` survives between calls. A local `window = []` created inside every method
would start empty each time and lose the history.

Before coding, answer five questions:

1. **API:** What does each method record or return?
2. **Identity:** Is state global, per user, per message, per route, or per key?
3. **Time:** Do timestamps arrive in order? Can old events be corrected?
4. **Retention:** What may be discarded, and exactly when?
5. **Invariant:** What must remain true after every call?

### Why a deque often appears

If timestamps arrive in order, the oldest event is at the front. Expiration happens in that
same order, so `deque.popleft()` removes the next expired event in O(1) time.
A heap becomes useful when the next item we need is selected by a priority such as price.
The required operation determines the structure.

### Two boundaries you must distinguish

| Active interval at time t | Event exactly at t − W | Expire while |
|---|---|---|
| `[t − W, t]` | Keep it | `old < t - W` |
| `(t − W, t]` | Remove it | `old <= t - W` |

Square brackets include an endpoint; parentheses exclude it. The contract determines the comparison.
Cleanup in these classes happens **when a method is called**; no background timer runs.
Expired records may remain between calls, but queries remove them before using the state.

Assume calls are sequential. The LeetCode classes use their stated valid-input guarantees.
The two custom limiters state their own contracts and validate their configuration and time order.
Dictionary operation costs below are expected O(1); list appends are amortized O(1).

<a id="guide"></a>

## The decision that connects these problems

**Ask what a future query could still need.** That decides what you may forget.

| Query needs… | Keep… | You may forget… |
|---|---|---|
| Next eligible time for a message | One cooldown per message | Earlier successful prints |
| Recent events or accepted requests | A deque of live timestamps | Events permanently outside the window |
| Recent count with repeated seconds | Timestamp/count buckets | Individual identities within a bucket |
| Average of completed trips | Sum and count per route | Individual completed trips |
| Historical values or log IDs | Versions or logs | Nothing unless the API gives a retention limit |
| Extremes while old entries can change | Current-value map and heaps | Obsolete heap entries when detected |
| Refillable allowance | Tokens and last refill time | Individual requests |

### Explain your design out loud

> “The API asks for ____. The key is ____. I store ____ because the next call needs ____.
> After each call, ____ must be true. I can discard ____ because ____.
> This makes each update cost ____ and each query cost ____.”

For the sliding API limiter:

> “I need to decide whether this client has room for one more accepted request.
> I keep a deque of accepted timestamps per client. With increasing time, expiry is always
> at the front. I remove timestamps at or before t − W, reject if the remaining count is L,
> otherwise append t. Each accepted timestamp is appended and removed at most once.”

### Four follow-ups that change your code

- **Out-of-order time:** A deque is no longer ordered by expiry. You need an ordered structure
  and a policy for late events; previously discarded history cannot answer arbitrary past queries.
- **Corrections/deletions:** Aggregates need the old contribution to subtract it. Preserve enough
  identity to undo a change, or maintain an authoritative map and validate derived indexes.
- **Unbounded identities:** Per-key bounded state still grows with the number of keys. Define
  when inactive keys can be removed and how cleanup is triggered.
- **Concurrent callers:** Checking and changing a quota must happen atomically. Locks or shared
  atomic storage belong around the whole operation, not just the final append.

### Variations

Every variation is the method again with one different structure, or a different answer to who pays. The table is the overview and a lookup: find the trick, then the problems that use it. The classes after it follow its order, and the first two, LFU Cache and Insert Delete GetRandom O(1), are the designs to know right after LRU.

| Variation | What changes from the template | Problems |
|---|---|---|
| **Count buckets + a minimum pointer** | the truth is key → count; each count has a bucket of keys, oldest first; `min_freq` only resets to 1 or steps up by 1 | LFU Cache (460, below): evict the least frequently used key, ties to the least recently used |
| **Dense list + index map** | a list with no holes answers the random pick; value → index answers membership; delete by moving the last value into the hole | Insert Delete GetRandom O(1) (380, below): insert, remove and a uniform random pick |
| **Heap + lazy deletion** | the dict is the truth; heap entries that disagree with it are skipped when they surface | Stock Price Fluctuation (2034, above); Number Container System (2349): `change(i, x)` fills slot i, `find(x)` is the smallest index holding x; Design a Food Rating System (2353): the best-rated food of a cuisine while ratings change |
| **A heap of free items** | the smallest free seat or id is the heap's top; freeing pushes it back | Seat Reservation Manager (1845): `reserve()` hands out the smallest free seat, `unreserve(s)` gives it back |
| **Counters per line** | row, column and both diagonal sums; a move updates at most 4 counters, a win is a counter reaching ±n | Design Tic-Tac-Toe (348): `move(row, col, player)` returns the winner after the move, or 0 |
| **Deque + set** | the snake's body is a deque (head in, tail out) plus a set for O(1) self-collision checks | Design Snake Game (353): `move(direction)` returns the score, or −1 when the snake hits a wall or itself |
| **Hashing by hand** | an array of short chains at `hash(key) % B`; double B and rehash when chains get long | Design HashMap (706): `put`, `get`, `remove` without built-in hash tables |
| **A cursor and a logical end** | back/forward are index arithmetic; visit overwrites and moves the end | Design Browser History (1472, above) |
| **A snapshot next to each entry** | push (value, min so far) together | Min Stack (155): `push`, `pop`, `top`, `getMin`, all O(1), in [Stacks & Queues](00_Topic_Index.ipynb#s07) |
| **Two heaps around the middle** | a max-heap for the low half, a min-heap for the high half, sizes within one | Find Median from Data Stream (295): `addNum`, `findMedian`, in [Heaps](15_Heaps.ipynb#topic-heaps) |
| **A flag and a maintained count** | flip-all toggles one flag; the count of ones is kept current on every write | Design Bitset (2166, below): fix, unfix, flip all and count, each O(1) |
| **Two numbers instead of a log** | tokens left and the time they were counted; each call adds the refill first | a token-bucket rate limiter (below), which allows short bursts |
| **A one-item buffer** | the next item is fetched ahead and kept beside a `done` flag | Peeking Iterator (284): `peek()` without consuming; Flatten Nested List Iterator (341); Zigzag Iterator (281) |
| *Second pass:* **One stack per frequency** | a value pushed c times sits on floors 1..c; pop the top of the highest floor | Maximum Frequency Stack (895, below): pop the most frequent value, the most recent on a tie |
| *Second pass:* **Sorted boundaries + bisect** | intervals stay disjoint and sorted; a change only touches its neighbours | Range Module (715, below): track, untrack and query half-open ranges; Data Stream as Disjoint Intervals (352): `addNum(v)`, `getIntervals()` as merged `[start, end]` pairs |
| *Second pass:* **Count buckets in a linked list** | LRU's list, but each node is a *count* holding a set of keys; ±1 moves a key to the neighbouring bucket | All O'one Data Structure (432): `inc` and `dec` a key's count, `getMaxKey` and `getMinKey`, all O(1) |
| *Second pass:* **Heap of candidates, validated lazily** | a min-heap of indices that may have room; check the top before trusting it | Dinner Plate Stacks (1172): `push` onto the leftmost stack with room, `pop` from the rightmost, `popAtStack(i)` |
| *Second pass:* **Heaps + version stamps** | every state change pushes a fresh entry; stale entries are skipped when they surface | Design Movie Rental System (1912): `search` the five cheapest unrented copies of a movie, `rent`, `drop`, `report` the five cheapest rented |
| *Second pass:* **Trie of dicts** | each path component is an edge; a node is a directory (children) or a file (content) | Design In-Memory File System (588): `ls`, `mkdir`, `addContentToFile`, `readContentFromFile` |
| *Second pass:* **Express lanes** | a sorted linked list plus random-height towers; a search drops a level when it cannot move right | Design Skiplist (1206): `search`, `add`, `erase` in O(log n) expected, no built-in ordered structure |
| *Second pass:* **Positions + random sampling** | value → sorted positions; sample a few indices, verify each with two bisects | Online Majority Element In Subarray (1157): `query(l, r, threshold)`, a value occurring at least `threshold` times in `arr[l..r]`, where `threshold` is more than half its length, or −1 |
| *Second pass:* **Fenwick tree** | point update and prefix sum in O(log n) per axis; a rectangle is 4 prefixes | Range Sum Query 2D - Mutable (308): `update(r, c, v)` and `sumRegion` of a rectangle, interleaved |

Three follow-ups come up again and again after any tracker. Memory: the Logger's dict keeps every message forever, and a deque of `(t, message)` beside it lets you delete entries older than 10 seconds. O(1)-memory windows: HitCounter's two 300-slot lists, in its Try it above. Concurrency: guard each public method with one `threading.Lock`, so no caller ever sees the truth and an index half-updated.